# N155 · SQL窗口、排名与连续段

**目标：** 区分窗口保留行和聚合折叠行。

**先修：** N154。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** ROW_NUMBER/RANK/DENSE_RANK；PARTITION；LAG/LEAD；窗口frame；gaps and islands。

## 从问题到算法

窗口函数在保留明细行的同时计算组内排序。ROW_NUMBER对每行编号，RANK并列后跳号，DENSE_RANK不跳号；前三种不同工资要用DENSE_RANK。连续段必须先定义连续的是行号、ID还是自然日期，本章Logs和Stadium按连续ID判断，ID缺口不能被忽略。

## 最小实现

**本章接口：** `query_dense_rank.sql`、`query_consecutive_numbers.sql`、`query_department_top3.sql`、`query_stadium.sql`

In [1]:
import sqlite3

SQL_QUERIES = {
    'query_dense_rank.sql': r'''SELECT score, DENSE_RANK() OVER (ORDER BY score DESC) AS rank
FROM Scores ORDER BY score DESC;''',
    'query_consecutive_numbers.sql': r'''SELECT DISTINCT a.num AS ConsecutiveNums
FROM Logs AS a
JOIN Logs AS b ON b.id = a.id + 1 AND b.num = a.num
JOIN Logs AS c ON c.id = a.id + 2 AND c.num = a.num;''',
    'query_department_top3.sql': r'''WITH ranked AS (
    SELECT e.*, DENSE_RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) AS salary_rank
    FROM Employee AS e
)
SELECT d.name AS Department, r.name AS Employee, r.salary AS Salary
FROM ranked AS r JOIN Department AS d ON d.id = r.departmentId
WHERE r.salary_rank <= 3;''',
    'query_stadium.sql': r'''WITH eligible AS (
    SELECT *, id - ROW_NUMBER() OVER (ORDER BY id) AS run_id
    FROM Stadium WHERE people >= 100
), long_runs AS (
    SELECT run_id FROM eligible GROUP BY run_id HAVING COUNT(*) >= 3
)
SELECT e.id, e.visit_date, e.people
FROM eligible AS e JOIN long_runs AS r ON e.run_id = r.run_id
ORDER BY e.visit_date;''',
}

def create_sqlite_fixture():
    connection=sqlite3.connect(':memory:')
    connection.executescript(r'''
CREATE TABLE Scores(id INTEGER PRIMARY KEY,score REAL);
INSERT INTO Scores VALUES (1,3.5),(2,3.65),(3,4.0),(4,3.85),(5,4.0),(6,3.65);
CREATE TABLE Logs(id INTEGER PRIMARY KEY,num INTEGER);
INSERT INTO Logs VALUES (1,7),(2,7),(3,7),(5,9),(6,9),(8,9),(10,2),(11,2),(12,2),(13,2);
CREATE TABLE Department(id INTEGER PRIMARY KEY,name TEXT);
CREATE TABLE Employee(id INTEGER PRIMARY KEY,name TEXT,salary INTEGER,departmentId INTEGER);
INSERT INTO Department VALUES (1,'D');
INSERT INTO Employee VALUES (1,'A',100,1),(2,'B',90,1),(3,'C',80,1),(4,'D',80,1),(5,'E',70,1);
CREATE TABLE Stadium(id INTEGER PRIMARY KEY,visit_date TEXT,people INTEGER);
INSERT INTO Stadium VALUES (1,'2021-01-01',100),(2,'2021-01-02',110),(3,'2021-01-03',120),
(5,'2021-01-05',100),(6,'2021-01-06',100),(7,'2021-01-07',100),(8,'2021-01-08',100),
(10,'2021-01-10',200),(11,'2021-01-11',200),(12,'2021-01-12',90),(13,'2021-01-13',200);
''')
    return connection

def query_rows(connection,name):
    return connection.execute(SQL_QUERIES[name]).fetchall()

## 正确性、前提与复杂度

过滤后的高人流记录按ID排序，相邻ID连续时id与row_number同步加1，差值保持不变；ID断开时差值变化，从而得到连续岛。先筛选再编号使不达标记录打断连续段。工资排名按部门分区，筛rank≤3保留第三高工资的所有并列员工。

**代价：** 窗口排序通常O(n log n)，合适索引可能减少排序代价；连续三ID自连接可利用主键索引。返回顺序只在查询明写ORDER BY时保证。SQLite窗口函数真实执行通过后才记录本章成功。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

db=create_sqlite_fixture()
for name in ['query_dense_rank.sql','query_stadium.sql']:
    cursor=db.execute(SQL_QUERIES[name]); print(name); show_table([x[0] for x in cursor.description],cursor.fetchall())
db.close()

query_dense_rank.sql


score,rank
4.0,1
4.0,1
3.85,2
3.65,3
3.65,3
3.5,4


query_stadium.sql


id,visit_date,people
1,2021-01-01,100
2,2021-01-02,110
3,2021-01-03,120
5,2021-01-05,100
6,2021-01-06,100
7,2021-01-07,100
8,2021-01-08,100


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
db=create_sqlite_fixture()
assert query_rows(db,'query_dense_rank.sql')==[(4.0,1),(4.0,1),(3.85,2),(3.65,3),(3.65,3),(3.5,4)]
assert set(query_rows(db,'query_consecutive_numbers.sql'))=={(7,),(2,)}
assert {row[1] for row in query_rows(db,'query_department_top3.sql')}=={'A','B','C','D'}
assert [row[0] for row in query_rows(db,'query_stadium.sql')]==[1,2,3,5,6,7,8]
db.execute('DELETE FROM Logs WHERE id=2'); assert set(query_rows(db,'query_consecutive_numbers.sql'))=={(2,)}
db.close()
print('N155: 所有本章断言通过')

N155: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 区分ROWS与RANGE的重复排序键效果。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 给连续ID和连续日期分别建模。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 178. Rank Scores（canonical）
- 180. Consecutive Numbers（canonical）
- 185. Department Top Three Salaries（canonical）
- 601. Human Traffic of Stadium（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。